# 07 SFT Experiment

Supervised fine-tuning notebook that starts from the pretrain checkpoint produced by `05-pretrain-exp.ipynb` and trains on daily conversation SFT examples generated from the daily-dialog dataset zip. The tokenizer/pretrain artifacts still come from `04-preprocess-exp.ipynb` so the SFT checkpoint stays compatible with the pretrain model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Run once per fresh Colab runtime if sentencepiece is missing.
try:
    import sentencepiece  # noqa: F401
except ImportError:
    %pip -q install sentencepiece


In [ ]:
from pathlib import Path
import gzip
import json
import shutil
from collections import Counter

DATA_ROOT = Path('/content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp')
DATASET_ROOT = Path('/content/drive/MyDrive/text_dataset/dataset')
PREP_DIR = DATA_ROOT / 'prepared'
PRETRAIN_MODEL_DIR = DATA_ROOT / 'manuka-pretrain-model'
SFT_MODEL_DIR = DATA_ROOT / 'manuka-sft-model'
LOCAL_PREP_DIR = Path('/content/manuka-07-sft-prepared')
DAILY_ZIP_NAME = '\uc77c\uc0c1\ub300\ud654_\ub370\uc774\ud130.zip'

EXPECTED_SFT_KIND = 'daily_dialog_sft'
SFT_PAIR_MODE = 'daily_single_turn_previous_to_next'
SFT_PAIRS_NAME = 'daily_dialog_sft_pairs.jsonl.gz'
SFT_TOKENIZED_NAME = 'daily_dialog_sft_tokenized_examples.jsonl.gz'
FORCE_REBUILD_SFT_DATA = False

if not PREP_DIR.exists():
    raise FileNotFoundError(f'{PREP_DIR} does not exist. Run 04-preprocess-exp.ipynb first.')
if not PRETRAIN_MODEL_DIR.exists():
    raise FileNotFoundError(f'{PRETRAIN_MODEL_DIR} does not exist. Run 05-pretrain-exp.ipynb first.')

pretrain_config_path = PRETRAIN_MODEL_DIR / 'config.json'
pretrain_model_path = PRETRAIN_MODEL_DIR / 'model.pt'
if not pretrain_config_path.exists():
    raise FileNotFoundError(f'{pretrain_config_path} not found.')
if not pretrain_model_path.exists():
    raise FileNotFoundError(f'{pretrain_model_path} not found.')

if LOCAL_PREP_DIR.exists():
    shutil.rmtree(LOCAL_PREP_DIR)
LOCAL_PREP_DIR.mkdir(parents=True, exist_ok=True)
SFT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

manifest_src = PREP_DIR / 'data_manifest.json'
if not manifest_src.exists():
    raise FileNotFoundError(f'{manifest_src} not found. Run preprocessing first.')
shutil.copy2(manifest_src, LOCAL_PREP_DIR / manifest_src.name)

with open(manifest_src, encoding='utf-8') as f:
    manifest = json.load(f)
with open(pretrain_config_path, encoding='utf-8') as f:
    cfg = json.load(f)

MAX_LEN = int(cfg['max_len'])
if int(manifest.get('context_length', MAX_LEN)) != MAX_LEN:
    raise ValueError(f"Prepared context length {manifest.get('context_length')} does not match pretrained max_len {MAX_LEN}")

def existing_path(value):
    if not value:
        return None
    path = Path(value)
    return path if path.exists() else None

_daily_zip_candidates = [
    existing_path(manifest.get('daily_zip_path')),
    DATASET_ROOT / DAILY_ZIP_NAME,
    DATA_ROOT / 'dataset' / DAILY_ZIP_NAME,
    DATA_ROOT / DAILY_ZIP_NAME,
]
DAILY_ZIP_PATH = next((p for p in _daily_zip_candidates if p is not None and p.exists()), None)
if DAILY_ZIP_PATH is None:
    searched = '\n'.join(str(p) for p in _daily_zip_candidates if p is not None)
    raise FileNotFoundError(f'Could not find {DAILY_ZIP_NAME}. Checked:\n{searched}')

tokenizer_name = cfg.get('tokenizer_model', manifest.get('tokenizer_model', 'spm32k.model'))
sft_pairs_name = SFT_PAIRS_NAME
sft_tokenized_name = SFT_TOKENIZED_NAME
sft_pairs_local_path = LOCAL_PREP_DIR / sft_pairs_name
sft_tokenized_local_path = LOCAL_PREP_DIR / sft_tokenized_name
sft_pairs_prepared_path = PREP_DIR / sft_pairs_name
sft_tokenized_prepared_path = PREP_DIR / sft_tokenized_name

print('Prepared data:', PREP_DIR)
print('Pretrain model:', PRETRAIN_MODEL_DIR)
print('Daily SFT zip:', DAILY_ZIP_PATH)
print('SFT model output:', SFT_MODEL_DIR)
print('SFT tokenized file:', sft_tokenized_name)
print('Base config:', {k: cfg[k] for k in ['vocab_size', 'd_model', 'n_layers', 'n_heads', 'n_kv_heads', 'max_len'] if k in cfg})


In [ ]:
import html
import io
import re
import zipfile
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file=str(PRETRAIN_MODEL_DIR / tokenizer_name))
PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id('<sep>')
VOCAB_SIZE = sp.get_piece_size()
SPECIALS = {'PAD': PAD, 'BOS': BOS, 'SEP': SEP, 'EOS': EOS, 'UNK': UNK}

if SEP < 0:
    raise ValueError('SFT requires the <sep> token from 04-preprocess-exp tokenizer.')
if VOCAB_SIZE != int(cfg['vocab_size']):
    raise ValueError(f"Tokenizer vocab size {VOCAB_SIZE} does not match pretrain config vocab_size {cfg['vocab_size']}")

TEXT_FIELD_PRIORITY = ('form', 'original_form', 'text', 'content', 'sentence', 'body')
WINDOW_STEPS = [1, 2]
ALLOW_SAME_SPEAKER_PAIRS = True
SPLIT_SENTENCES = True
MIN_TEXT_CHARS = 1
MIN_HANGUL_RATIO = 0.02
MAX_TEXT_CHARS = 512
MAX_PROMPT_CHARS = 512
MAX_RESPONSE_CHARS = 512

SPACE_RGX = re.compile(r'\s+')
TAG_RGX = re.compile(r'<[^>]+>')
KOR_RGX = re.compile('[\uac00-\ud7a3]')
SENT_RGX = re.compile(r'(?<=[.!?])\s+')
DECODINGS = ('utf-8-sig', 'utf-8', 'cp949', 'euc-kr')


def clean_text(s: str) -> str:
    s = html.unescape(str(s or ''))
    s = TAG_RGX.sub(' ', s)
    s = s.replace('​', ' ').replace('﻿', ' ').strip()
    return SPACE_RGX.sub(' ', s)


def is_usable_text(s: str) -> bool:
    s = clean_text(s)
    if len(s) < MIN_TEXT_CHARS:
        return False
    hangul = len(KOR_RGX.findall(s))
    return hangul >= max(1, int(len(s) * MIN_HANGUL_RATIO))


def pick_text(obj) -> str:
    if isinstance(obj, str):
        return clean_text(obj)
    if not isinstance(obj, dict):
        return ''
    for field in TEXT_FIELD_PRIORITY:
        text = clean_text(obj.get(field, ''))
        if text:
            return text
    return ''


def split_text(s: str):
    s = clean_text(s)
    if not s:
        return []
    parts = [p.strip() for p in SENT_RGX.split(s) if p.strip()] if SPLIT_SENTENCES else [s]
    out = []
    for part in parts or [s]:
        if len(part) <= MAX_TEXT_CHARS:
            out.append(part)
            continue
        current, current_len = [], 0
        for word in part.split():
            add_len = len(word) + (1 if current else 0)
            if current and current_len + add_len > MAX_TEXT_CHARS:
                out.append(' '.join(current))
                current, current_len = [word], len(word)
            else:
                current.append(word)
                current_len += add_len
        if current:
            out.append(' '.join(current))
    return [clean_text(x) for x in out if is_usable_text(x)]


def decode_bytes(raw: bytes):
    for encoding in DECODINGS:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            pass
    return raw.decode('utf-8', errors='replace'), 'utf-8-replace'


def load_json_member(zf, name):
    text, encoding = decode_bytes(zf.read(name))
    return json.loads(text), encoding


def iter_documents(data):
    if isinstance(data, dict):
        docs = data.get('document')
        if isinstance(docs, list):
            yield from docs
            return
        if isinstance(docs, dict):
            yield docs
            return
        yield data
    elif isinstance(data, list):
        yield from data


def utterance_sort_key(utt):
    start = utt.get('start') if isinstance(utt, dict) else None
    if isinstance(start, str):
        try:
            start = float(start)
        except ValueError:
            start = float('inf')
    if not isinstance(start, (int, float)):
        start = float('inf')
    return (start, str(utt.get('id', '')) if isinstance(utt, dict) else '')


def encode_text(s: str):
    return sp.encode(clean_text(s), out_type=int)


def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([int(i) for i in ids if int(i) not in drop])


def encode_sft_pair(pair):
    prompt_ids = encode_text(pair['prompt'])
    response_ids = encode_text(pair['response']) + [EOS]

    max_response_tokens = max(8, MAX_LEN - 32)
    if len(response_ids) > max_response_tokens:
        response_ids = response_ids[:max_response_tokens]
        response_ids[-1] = EOS

    prompt_room = MAX_LEN - len(response_ids) - 2
    if prompt_room <= 0:
        return None
    prompt_ids = prompt_ids[-prompt_room:]

    input_ids = [BOS] + prompt_ids + [SEP] + response_ids
    labels = [-100] * (len(prompt_ids) + 2) + response_ids
    if len(input_ids) <= MAX_LEN and any(t != -100 for t in labels):
        return {
            'input_ids': input_ids,
            'labels': labels,
            'kind': pair.get('kind', EXPECTED_SFT_KIND),
            'doc_id': pair.get('doc_id', ''),
            'source': pair.get('source', ''),
            'prompt_speaker': pair.get('prompt_speaker', ''),
            'response_speaker': pair.get('response_speaker', ''),
        }
    return None


def iter_daily_sft_pairs(zip_path):
    stats = Counter()
    bad_files = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith('.json'):
                continue
            stats['json_files'] += 1
            try:
                data, _ = load_json_member(zf, name)
                for doc in iter_documents(data):
                    if not isinstance(doc, dict):
                        continue
                    utterances = doc.get('utterance', [])
                    if not isinstance(utterances, list):
                        continue
                    stats['documents'] += 1
                    doc_id = str(doc.get('id', ''))
                    units = []
                    for utt in sorted(utterances, key=utterance_sort_key):
                        if not isinstance(utt, dict):
                            continue
                        stats['utterances_seen'] += 1
                        speaker = str(utt.get('speaker_id') or utt.get('speaker') or 'unknown')
                        for text in split_text(pick_text(utt)):
                            units.append({'text': text[:MAX_TEXT_CHARS], 'speaker': speaker})
                    stats['usable_units'] += len(units)
                    if len(units) < 2:
                        continue
                    for step in WINDOW_STEPS:
                        for i in range(len(units) - step):
                            left, right = units[i], units[i + step]
                            if not ALLOW_SAME_SPEAKER_PAIRS and left['speaker'] == right['speaker']:
                                continue
                            prompt = clean_text(left['text'])[:MAX_PROMPT_CHARS]
                            response = clean_text(right['text'])[:MAX_RESPONSE_CHARS]
                            if not is_usable_text(prompt) or not is_usable_text(response):
                                continue
                            stats[f'pairs_step_{step}'] += 1
                            yield {
                                'prompt': prompt,
                                'response': response,
                                'prompt_speaker': left['speaker'],
                                'response_speaker': right['speaker'],
                                'kind': EXPECTED_SFT_KIND,
                                'doc_id': doc_id,
                                'source': name,
                            }, stats, bad_files
            except Exception as exc:
                bad_files.append({'kind': EXPECTED_SFT_KIND, 'source': name, 'error': repr(exc)})
    stats['bad_files'] = len(bad_files)
    yield None, stats, bad_files


def load_tokenized_sft(path):
    rows = []
    kind_counts = Counter()
    tokens_per_epoch = 0
    response_tokens_per_epoch = 0
    max_seen_len = 0
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            row = json.loads(line)
            kind = row.get('kind', EXPECTED_SFT_KIND)
            if kind != EXPECTED_SFT_KIND:
                continue
            ids = list(row['input_ids'])
            labels = list(row['labels'])
            if len(ids) != len(labels):
                raise ValueError(f'input_ids/labels length mismatch on row {line_no}')
            if len(ids) > MAX_LEN:
                ids = ids[:MAX_LEN]
                labels = labels[:MAX_LEN]
            if len(ids) < 2 or not any(t != -100 for t in labels):
                continue
            rows.append({
                'input_ids': ids,
                'labels': labels,
                'kind': kind,
                'doc_id': row.get('doc_id', ''),
                'source': row.get('source', ''),
                'prompt_speaker': row.get('prompt_speaker', ''),
                'response_speaker': row.get('response_speaker', ''),
            })
            kind_counts[kind] += 1
            tokens_per_epoch += len(ids)
            response_tokens_per_epoch += sum(1 for t in labels if t != -100)
            max_seen_len = max(max_seen_len, len(ids))
            if line_no % 250000 == 0:
                print(f'read {line_no:,} rows | kept {len(rows):,}')
    return rows, kind_counts, tokens_per_epoch, response_tokens_per_epoch, max_seen_len


if FORCE_REBUILD_SFT_DATA or not sft_tokenized_prepared_path.exists():
    build_stats = Counter()
    bad_files = []
    sft_pair_count = 0
    sft_tokenized_count = 0
    sft_tokens_written = 0
    sft_max_written_len = 0
    with gzip.open(sft_pairs_local_path, 'wt', encoding='utf-8') as pair_f, gzip.open(sft_tokenized_local_path, 'wt', encoding='utf-8') as tok_f:
        for pair, stats, current_bad_files in iter_daily_sft_pairs(DAILY_ZIP_PATH):
            build_stats = stats
            bad_files = current_bad_files
            if pair is None:
                continue
            sft_pair_count += 1
            pair_f.write(json.dumps(pair, ensure_ascii=False) + '\n')
            row = encode_sft_pair(pair)
            if row is None:
                continue
            tok_f.write(json.dumps(row, ensure_ascii=False) + '\n')
            sft_tokenized_count += 1
            sft_tokens_written += len(row['input_ids'])
            sft_max_written_len = max(sft_max_written_len, len(row['input_ids']))
            if sft_pair_count % 250000 == 0:
                print(f'built {sft_pair_count:,} pairs | tokenized {sft_tokenized_count:,}')

    if sft_tokenized_count == 0:
        raise ValueError('No daily-dialog SFT examples were produced.')
    shutil.copy2(sft_pairs_local_path, sft_pairs_prepared_path)
    shutil.copy2(sft_tokenized_local_path, sft_tokenized_prepared_path)
else:
    print(f'Using cached daily SFT examples: {sft_tokenized_prepared_path}')
    shutil.copy2(sft_tokenized_prepared_path, sft_tokenized_local_path)
    if sft_pairs_prepared_path.exists():
        shutil.copy2(sft_pairs_prepared_path, sft_pairs_local_path)
    build_stats = Counter({'cached': 1})
    bad_files = []

sft_examples, kind_counts, tokens_per_epoch, response_tokens_per_epoch, max_seen_len = load_tokenized_sft(sft_tokenized_local_path)
if not sft_examples:
    raise ValueError('No SFT examples were loaded.')

sft_data_manifest = {
    'phase': 'sft',
    'kind': EXPECTED_SFT_KIND,
    'pair_mode': SFT_PAIR_MODE,
    'source_zip': str(DAILY_ZIP_PATH),
    'pairs_file': sft_pairs_name,
    'tokenized_file': sft_tokenized_name,
    'examples': len(sft_examples),
    'tokens_per_epoch_before_padding': tokens_per_epoch,
    'response_tokens_per_epoch': response_tokens_per_epoch,
    'max_token_length': max_seen_len,
    'window_steps': WINDOW_STEPS,
    'allow_same_speaker_pairs': ALLOW_SAME_SPEAKER_PAIRS,
    'split_sentences': SPLIT_SENTENCES,
    'build_stats': dict(build_stats),
    'bad_file_count': len(bad_files),
    'bad_files_sample': bad_files[:50],
}
with open(LOCAL_PREP_DIR / 'sft_data_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(sft_data_manifest, f, ensure_ascii=False, indent=2)
shutil.copy2(LOCAL_PREP_DIR / 'sft_data_manifest.json', PREP_DIR / 'sft_data_manifest.json')

print('Tokenizer vocab size:', VOCAB_SIZE, SPECIALS)
print('SFT rows:', len(sft_examples))
print('SFT row mix:', dict(kind_counts))
print('Tokens per epoch before dynamic padding:', tokens_per_epoch)
print('Response label tokens per epoch:', response_tokens_per_epoch)
print('Max token length:', max_seen_len)
print('Daily SFT manifest:', PREP_DIR / 'sft_data_manifest.json')


In [ ]:
first = sft_examples[0]
prompt_preview_ids = []
response_preview_ids = []
seen_sep = False
for token_id, label_id in zip(first['input_ids'], first['labels']):
    if token_id == SEP:
        seen_sep = True
        continue
    if not seen_sep:
        prompt_preview_ids.append(token_id)
    elif label_id != -100:
        response_preview_ids.append(label_id)

print('First SFT row token length:', len(first['input_ids']))
print('First SFT prompt preview:', decode_text(prompt_preview_ids[:120]))
print('First SFT response preview:', decode_text(response_preview_ids[:120]))


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint


def rotate_half(x):
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


def apply_rope(q, k, *, rope_freqs, start_pos=0):
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("q and k must have the same head dimension")

    original_dim = q.shape[-1]
    head_dim = original_dim + original_dim % 2
    if original_dim % 2 != 0:
        q = F.pad(q, (0, 1))
        k = F.pad(k, (0, 1))

    seq_len = q.shape[-2]
    freqs = rope_freqs[start_pos:start_pos + seq_len, :head_dim].to(device=q.device, dtype=torch.float32)
    cos = freqs.cos().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)
    sin = freqs.sin().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)

    q_out = (q * cos) + (rotate_half(q) * sin)
    k_out = (k * cos) + (rotate_half(k) * sin)
    if original_dim % 2 != 0:
        q_out = q_out[..., :original_dim]
        k_out = k_out[..., :original_dim]
    return q_out.contiguous(), k_out.contiguous()


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        normed = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return normed.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)


class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        dropout,
        max_len,
        n_kv_heads=None,
        qk_norm=True,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.n_kv_heads = n_heads if n_kv_heads is None else n_kv_heads
        if n_heads % self.n_kv_heads != 0:
            raise ValueError("n_heads must be divisible by n_kv_heads")
        if rope_scale <= 0:
            raise ValueError("rope_scale must be positive")
        if rope_scaling_type not in {"linear", "ntk"}:
            raise ValueError("rope_scaling_type must be 'linear' or 'ntk'")

        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // self.n_kv_heads
        self.rope_theta = rope_theta
        self.rope_scale = rope_scale
        self.rope_scaling_type = rope_scaling_type

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("rope_freqs", self._build_rope_freqs(max_len), persistent=False)

    def _build_rope_freqs(self, max_len):
        head_dim = self.head_dim + self.head_dim % 2
        fractions = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        inv_freq = 1.0 / (self.rope_theta ** fractions)
        if self.rope_scaling_type == "ntk":
            inv_freq = inv_freq / (self.rope_scale ** fractions)
            positions = torch.arange(max_len, dtype=torch.float32)
        else:
            positions = torch.arange(max_len, dtype=torch.float32) / self.rope_scale
        freqs = torch.einsum("i,j->ij", positions, inv_freq)
        return torch.repeat_interleave(freqs, 2, dim=1)

    def _repeat_kv(self, x):
        if self.kv_repeat == 1:
            return x
        return x.repeat_interleave(self.kv_repeat, dim=1)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        past_len = past_kv[0].shape[-2] if past_kv is not None else 0
        start_pos = past_len
        q, k = apply_rope(q, k, rope_freqs=self.rope_freqs, start_pos=start_pos)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        present = (k, v) if use_cache else None

        k_full = self._repeat_kv(k)
        v_full = self._repeat_kv(v)
        key_len = k_full.shape[-2]

        causal_needed = T > 1 or past_kv is None
        attn_bias = None
        if attn_mask is not None:
            if attn_mask.dim() != 2:
                raise ValueError("attn_mask must have shape (batch, key_length)")
            if attn_mask.shape[-1] != key_len:
                attn_mask = attn_mask[:, -key_len:]
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            attn_bias = torch.zeros((B, 1, T, key_len), device=x.device, dtype=q.dtype)
            attn_bias = attn_bias.masked_fill(pad_mask, torch.finfo(q.dtype).min)
            if causal_needed:
                q_pos = torch.arange(start_pos, start_pos + T, device=x.device).view(T, 1)
                k_pos = torch.arange(key_len, device=x.device).view(1, key_len)
                causal_mask = k_pos > q_pos
                attn_bias = attn_bias.masked_fill(causal_mask.view(1, 1, T, key_len), torch.finfo(q.dtype).min)
                causal_needed = False

        y = F.scaled_dot_product_attention(
            q,
            k_full,
            v_full,
            attn_mask=attn_bias,
            dropout_p=self.attn_drop.p if self.training else 0.0,
            is_causal=causal_needed,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.out_proj(y))
        return y, present


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        mlp_ratio=8/3,
        dropout=0.1,
        max_len=MAX_LEN,
        n_kv_heads=None,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        hidden_dim = int(mlp_ratio * d_model)
        hidden_dim = 256 * math.ceil(hidden_dim / 256)
        self.ln1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model,
            n_heads,
            dropout,
            max_len,
            n_kv_heads=n_kv_heads,
            qk_norm=qk_norm,
            rope_theta=rope_theta,
            rope_scale=rope_scale,
            rope_scaling_type=rope_scaling_type,
        )
        self.ln2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, hidden_dim, dropout)
        self.attn_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))
        self.mlp_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln1(x), attn_mask=attn_mask, past_kv=past_kv, use_cache=use_cache)
        x = x + self.attn_res_scale * attn_out
        x = x + self.mlp_res_scale * self.mlp(self.ln2(x))
        return x, present


class GPTScratch(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=640,
        n_layers=12,
        n_heads=10,
        n_kv_heads=10,
        max_len=MAX_LEN,
        dropout=0.1,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
        gradient_checkpointing=False,
    ):
        super().__init__()
        self.max_len = max_len
        self.n_layers = n_layers
        self.gradient_checkpointing = gradient_checkpointing
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_norm = RMSNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                mlp_ratio=8/3,
                dropout=dropout,
                max_len=max_len,
                n_kv_heads=n_kv_heads,
                qk_norm=qk_norm,
                residual_scale=residual_scale,
                rope_theta=rope_theta,
                rope_scale=rope_scale,
                rope_scaling_type=rope_scaling_type,
            )
            for _ in range(n_layers)
        ])
        self.ln_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init)
        self._scale_residual_projections()
        self.head.weight = self.tok_emb.weight

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _scale_residual_projections(self):
        scale = 0.02 / math.sqrt(2 * self.n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scale)
            nn.init.normal_(block.mlp.w3.weight, mean=0.0, std=scale)

    def forward(self, x, attention_mask=None, past_kv=None, use_cache=False):
        B, T = x.shape
        if past_kv is None and T > self.max_len:
            x = x[:, -self.max_len:]
            T = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len:]

        h = self.drop(self.emb_norm(self.tok_emb(x)))
        presents = [] if use_cache else None
        if past_kv is None:
            past_kv = [None] * len(self.blocks)

        for block, layer_past in zip(self.blocks, past_kv):
            if self.gradient_checkpointing and self.training and not use_cache:
                h, present = checkpoint.checkpoint(block, h, attention_mask, layer_past, False, use_reentrant=False)
            else:
                h, present = block(h, attn_mask=attention_mask, past_kv=layer_past, use_cache=use_cache)
            if use_cache:
                presents.append(present)

        logits = self.head(self.ln_f(h))
        if use_cache:
            return logits, presents
        return logits


In [ ]:
import os
import random
import time
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MODEL_DROPOUT = float(cfg.get('dropout', 0.08))
GRADIENT_CHECKPOINTING = False

model = GPTScratch(
    vocab_size=cfg['vocab_size'],
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    n_heads=cfg['n_heads'],
    n_kv_heads=cfg.get('n_kv_heads', cfg['n_heads']),
    max_len=cfg['max_len'],
    dropout=MODEL_DROPOUT,
    qk_norm=cfg.get('qk_norm', True),
    residual_scale=cfg.get('residual_scale', 1.0),
    rope_theta=cfg.get('rope_theta', 10000.0),
    rope_scale=cfg.get('rope_scale', 1.0),
    rope_scaling_type=cfg.get('rope_scaling_type', 'linear'),
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)
state_dict = torch.load(pretrain_model_path, map_location='cpu')
model.load_state_dict(state_dict, strict=True)
model = model.to(device)

BASE_LR = 2e-5
MIN_LR = 2e-6
WEIGHT_DECAY = 0.02
LABEL_SMOOTHING = 0.0
BATCH_SIZE = 128
ACCUM = 4
EPOCHS = 1
PATIENCE = 2
VAL_FRACTION = 0.02
EVAL_MAX_BATCHES = None
USE_COMPILE = False
save_dir = str(SFT_MODEL_DIR)

param_count = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Loaded pretrained parameters: {param_count / 1e6:.1f}M")
print({
    'vocab_size': cfg['vocab_size'],
    'd_model': cfg['d_model'],
    'n_layers': cfg['n_layers'],
    'n_heads': cfg['n_heads'],
    'n_kv_heads': cfg.get('n_kv_heads', cfg['n_heads']),
    'max_len': cfg['max_len'],
})

if USE_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model)
        print("torch.compile enabled")
    except Exception as exc:
        print("torch.compile skipped:", repr(exc))

optimizer_kwargs = dict(lr=BASE_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95), eps=1e-8)
if device == "cuda":
    optimizer_kwargs["fused"] = True
try:
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)
except TypeError:
    optimizer_kwargs.pop("fused", None)
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)

amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))


def lm_loss(logits, targets):
    vocab = logits.size(-1)
    pred = logits[:, :-1].contiguous().view(-1, vocab)
    gold = targets[:, 1:].contiguous().view(-1)
    return F.cross_entropy(pred, gold, ignore_index=-100, label_smoothing=LABEL_SMOOTHING)


class SFTDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return row["input_ids"], row["labels"]


def collate_batch(batch, pad_to_multiple_of=8):
    max_len = max(len(x[0]) for x in batch)
    if pad_to_multiple_of:
        max_len = int(math.ceil(max_len / pad_to_multiple_of) * pad_to_multiple_of)
    input_ids, labels, attention_mask = [], [], []
    for ids, lbl in batch:
        pad_len = max_len - len(ids)
        input_ids.append(ids + [PAD] * pad_len)
        labels.append(lbl + [-100] * pad_len)
        attention_mask.append([1] * len(ids) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

rows = sft_examples[:]
rng = random.Random(SEED)
rng.shuffle(rows)
val_size = max(1, int(len(rows) * VAL_FRACTION))
val_rows = rows[:val_size]
train_rows = rows[val_size:]

train_dataset = SFTDataset(train_rows)
val_dataset = SFTDataset(val_rows)
loader_kwargs = dict(
    num_workers=4 if device == "cuda" else 0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_batch,
)
if loader_kwargs["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, **loader_kwargs)
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

updates_per_epoch = math.ceil(len(train_dl) / ACCUM)
TOTAL_STEPS = EPOCHS * updates_per_epoch
WARMUP_STEPS = max(25, int(0.03 * TOTAL_STEPS))


def lr_lambda(step):
    min_ratio = MIN_LR / BASE_LR
    if step < WARMUP_STEPS:
        return max(min_ratio, float(step + 1) / float(WARMUP_STEPS))
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return min_ratio + (1.0 - min_ratio) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(max_batches=None):
    model.eval(); total_loss, total_tokens = 0.0, 0
    for bi, b in enumerate(val_dl, 1):
        x = b["input_ids"].to(device, non_blocking=True)
        y = b["labels"].to(device, non_blocking=True)
        attn = b["attention_mask"].to(device, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            loss = lm_loss(model(x, attention_mask=attn), y)
        target_tokens = (y[:, 1:] != -100).sum().item()
        total_loss += loss.item() * max(1, target_tokens)
        total_tokens += max(1, target_tokens)
        if max_batches and bi >= max_batches:
            break
    return total_loss / max(1, total_tokens)


def raw_model_state_dict():
    raw = getattr(model, "_orig_mod", model)
    return raw.state_dict()

print(f"Train examples: {len(train_dataset):,} | Val examples: {len(val_dataset):,} | Updates/epoch: {updates_per_epoch:,}")

best = float("inf")
bad_epochs = 0
os.makedirs(save_dir, exist_ok=True)
for src_name in [tokenizer_name, Path(tokenizer_name).with_suffix('.vocab').name, 'data_manifest.json', 'sft_data_manifest.json']:
    if src_name == 'sft_data_manifest.json':
        src_path = LOCAL_PREP_DIR / src_name
    elif src_name == 'data_manifest.json':
        src_path = PREP_DIR / src_name
    else:
        src_path = PRETRAIN_MODEL_DIR / src_name
    if src_path.exists():
        shutil.copy2(src_path, Path(save_dir) / src_name)

for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); optimizer.zero_grad(set_to_none=True)
    running_loss, running_count = 0.0, 0
    for i, b in enumerate(train_dl, 1):
        x = b["input_ids"].to(device, non_blocking=True)
        y = b["labels"].to(device, non_blocking=True)
        attn = b["attention_mask"].to(device, non_blocking=True)
        group_start = ((i - 1) // ACCUM) * ACCUM + 1
        group_end = min(group_start + ACCUM - 1, len(train_dl))
        accum_scale = group_end - group_start + 1
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            loss = lm_loss(model(x, attention_mask=attn), y) / accum_scale
        scaler.scale(loss).backward()
        running_loss += loss.item() * accum_scale
        running_count += 1
        if i % ACCUM == 0 or i == len(train_dl):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step = (ep - 1) * updates_per_epoch + math.ceil(i / ACCUM)
            if step % 100 == 0:
                print(f"step {step:,} | loss {running_loss / max(1, running_count):.4f} | lr {scheduler.get_last_lr()[0]:.2e}")
                running_loss, running_count = 0.0, 0

    val = evaluate(max_batches=EVAL_MAX_BATCHES)
    print(f"[epoch {ep}] val_loss={val:.4f} | {(time.time() - t0) / 60:.1f} min")
    if val < best:
        best = val
        bad_epochs = 0
        torch.save(raw_model_state_dict(), f"{save_dir}/model.pt")
        with open(f"{save_dir}/config.json", "w", encoding="utf-8") as f:
            json.dump({
                "vocab_size": cfg['vocab_size'],
                "d_model": cfg['d_model'],
                "n_layers": cfg['n_layers'],
                "n_heads": cfg['n_heads'],
                "n_kv_heads": cfg.get('n_kv_heads', cfg['n_heads']),
                "max_len": cfg['max_len'],
                "dropout": MODEL_DROPOUT,
                "qk_norm": cfg.get('qk_norm', True),
                "residual_scale": cfg.get('residual_scale', 1.0),
                "rope_theta": cfg.get('rope_theta', 10000.0),
                "rope_scale": cfg.get('rope_scale', 1.0),
                "rope_scaling_type": cfg.get('rope_scaling_type', 'linear'),
                "gradient_checkpointing": GRADIENT_CHECKPOINTING,
                "special_tokens": SPECIALS,
                "tokenizer_model": tokenizer_name,
                "generation": {"max_new_tokens": 192, "temperature": 0.72, "top_p": 0.88, "top_k": 60, "repetition_penalty": 1.12, "no_repeat_ngram_size": 4},
                "optimizer": {"base_lr": BASE_LR, "min_lr": MIN_LR, "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING},
                "training": {"phase": "sft", "epochs": EPOCHS, "batch_size": BATCH_SIZE, "accum": ACCUM, "warmup_steps": WARMUP_STEPS, "val_fraction": VAL_FRACTION, "best_val_loss": best},
                "base_model": {"model_dir": str(PRETRAIN_MODEL_DIR), "config": str(pretrain_config_path)},
                "sft_data": {"file": sft_tokenized_name, "pairs_file": sft_pairs_name, "kind": EXPECTED_SFT_KIND, "pair_mode": SFT_PAIR_MODE, "source_zip": str(DAILY_ZIP_PATH), "examples": len(sft_examples), "tokens_per_epoch_before_padding": tokens_per_epoch, "response_tokens_per_epoch": response_tokens_per_epoch},
                "paths": {"data_root": str(DATA_ROOT), "prepared_dir": str(PREP_DIR), "model_dir": str(SFT_MODEL_DIR)},
            }, f, ensure_ascii=False, indent=2)
        print("SAVED:", save_dir)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stop after {PATIENCE} epochs without validation improvement")
            break


In [ ]:
from pathlib import Path
import shutil

src_dir = SFT_MODEL_DIR
assert src_dir.is_dir(), f'{src_dir} folder does not exist. Train or copy a model there first.'

zip_base = DATA_ROOT / 'manuka-sft-model'
zip_file = Path(f'{zip_base}.zip')
if zip_file.exists():
    zip_file.unlink()

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format='zip',
    root_dir=src_dir.parent,
    base_dir=src_dir.name,
)
print('Saved zip:', zip_path)
